In [ ]:
from pathlib import Path

ROOT = Path(r"<PROJECT_ROOT>")

print("RouteSense exists:", ROOT.exists())
print("RouteSense path:", ROOT)

folders = [
    ROOT / "data" / "raw",
    ROOT / "data" / "cleaned",
    ROOT / "data" / "synthetic",
    ROOT / "sql",
    ROOT / "mongodb",
    ROOT / "notebooks"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("\nFolders created:")
for folder in folders:
    print(folder)

print("\nRaw folder exists:", (ROOT / "data" / "raw").exists())

In [ ]:
RAW = ROOT / "data" / "raw"

for file in RAW.iterdir():
    print(file.name)

In [ ]:
import pandas as pd

orders = pd.read_csv(
    RAW / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

customers = pd.read_csv(
    RAW / "olist_customers_dataset.csv"
)

print("Orders shape    :", orders.shape)
print("Customers shape :", customers.shape)

print("\nOrder columns:")
print(orders.columns.tolist())

print("\nCustomer columns:")
print(customers.columns.tolist())

print("\nOrder status counts:")
print(orders["order_status"].value_counts())

print("\nFirst 5 orders:")
print(orders.head())

In [ ]:
orders_master = orders.merge(
    customers[
        [
            "customer_id",
            "customer_city",
            "customer_state",
            "customer_zip_code_prefix"
        ]
    ],
    on="customer_id",
    how="left"
)

print("Orders before merge :", len(orders))
print("Rows after merge    :", len(orders_master))
print("Columns after merge :", len(orders_master.columns))
print("Master shape        :", orders_master.shape)

print("\nMaster columns:")
print(orders_master.columns.tolist())

print("\nFirst 5 rows:")
print(orders_master.head())

In [ ]:
orders_master["delivery_delay_days"] = (
    orders_master["order_delivered_customer_date"]
    - orders_master["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

print(
    orders_master["delivery_delay_days"]
    .describe()
)

print("\nSample delay values:")
print(
    orders_master[
        [
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_delay_days"
        ]
    ].head(10)
)

In [ ]:
def classify_delivery(row):
    if row["order_status"] == "canceled":
        return "CANCELLED"

    if row["order_status"] == "unavailable":
        return "UNAVAILABLE"

    if row["order_status"] == "delivered":
        if (
            pd.notna(row["delivery_delay_days"])
            and row["delivery_delay_days"] > 0
        ):
            return "LATE"
        return "ON_TIME"

    return "IN_PROGRESS"


orders_master["delivery_performance"] = orders_master.apply(
    classify_delivery,
    axis=1
)

print(
    orders_master["delivery_performance"]
    .value_counts()
)

print("\nPercentages:")
print(
    orders_master["delivery_performance"]
    .value_counts(normalize=True).mul(100).round(2)
)

In [ ]:
orders_master["delivery_failure"] = (
    orders_master["delivery_performance"]
    .isin(["LATE", "CANCELLED", "UNAVAILABLE"])
    .astype(int)
)

print("0 = No failure")
print("1 = Failure")

print("\nFailure counts:")
print(orders_master["delivery_failure"].value_counts())

print("\nFailure percentage:")
print(
    orders_master["delivery_failure"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
orders_master["delivery_duration_days"] = (
    orders_master["order_delivered_customer_date"]
    - orders_master["order_purchase_timestamp"]
).dt.total_seconds() / 86400

print(
    orders_master["delivery_duration_days"]
    .describe()
)

print("\nSample values:")
print(
    orders_master[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "delivery_duration_days"
        ]
    ].head(10)
)

In [ ]:
print("\n1. DATASET SHAPE")
print("Rows    :", orders_master.shape[0])
print("Columns :", orders_master.shape[1])

print("\n2. DELIVERY PERFORMANCE")
print(
    orders_master["delivery_performance"]
    .value_counts()
)

print("\n3. DELIVERY FAILURE")
print(
    orders_master["delivery_failure"]
    .value_counts()
)

print("\n4. FAILURE RATE")
failure_rate = orders_master["delivery_failure"].mean() * 100
print(f"{failure_rate:.2f}%")

print("\n5. MISSING VALUES")
missing = orders_master.isna().sum()
print(missing[missing > 0])

print("\n6. DUPLICATE ORDER IDs")
print(
    orders_master["order_id"].duplicated().sum()
)

print("\n7. UNIQUE ORDERS")
print(
    orders_master["order_id"].nunique()
)

print("\n8. FINAL COLUMNS")
for i, col in enumerate(orders_master.columns, start=1):
    print(f"{i}. {col}")

print("\n9. SAMPLE DATA")
print(orders_master.head())


In [ ]:
CLEANED = ROOT / "data" / "cleaned"
CLEANED.mkdir(parents=True, exist_ok=True)

output_file = CLEANED / "routesense_orders.csv"

orders_master.to_csv(
    output_file,
    index=False
)

print("File:", output_file)
print("Exists:", output_file.exists())

if output_file.exists():
    saved = pd.read_csv(output_file)
    print("Saved shape:", saved.shape)

In [ ]:
import pandas as pd
import numpy as np

cleaned_file = ROOT / "data" / "cleaned" / "routesense_orders.csv"

orders = pd.read_csv(cleaned_file)

print("\n1. CLEANED DATA LOADED")
print("Rows:", len(orders))
print("Columns:", len(orders.columns))


rng = np.random.default_rng(42)

num_drivers = 250

drivers = pd.DataFrame({
    "driver_id": [f"D{i:04d}" for i in range(1, num_drivers + 1)],
    "driver_name": [f"Driver_{i:04d}" for i in range(1, num_drivers + 1)],
    "experience_years": rng.integers(1, 16, num_drivers),
    "status": rng.choice(
        ["ACTIVE", "ON_LEAVE"],
        size=num_drivers,
        p=[0.92, 0.08]
    )
})

print("\n2. DRIVERS CREATED")
print("Drivers:", len(drivers))
print(drivers.head())

vehicles = pd.DataFrame({
    "vehicle_id": [f"V{i:04d}" for i in range(1, num_drivers + 1)],
    "vehicle_type": rng.choice(
        ["VAN", "BIKE", "TRUCK"],
        size=num_drivers,
        p=[0.45, 0.45, 0.10]
    ),
    "capacity_kg": rng.choice(
        [50, 100, 250, 500, 1000],
        size=num_drivers
    ),
    "status": rng.choice(
        ["AVAILABLE", "MAINTENANCE"],
        size=num_drivers,
        p=[0.95, 0.05]
    )
})

vehicles["driver_id"] = drivers["driver_id"]

print("\n3. VEHICLES CREATED")
print("Vehicles:", len(vehicles))
print(vehicles.head())

states = sorted(orders["customer_state"].dropna().unique())

route_records = []

route_number = 1

for state in states:
    for zone in range(1, 11):
        route_records.append({
            "route_id": f"R{route_number:04d}",
            "state": state,
            "zone_id": f"{state}-Z{zone:02d}",
            "planned_distance_km": round(
                float(rng.uniform(5, 60)), 2
            ),
            "planned_time_minutes": int(
                rng.integers(20, 180)
            )
        })
        route_number += 1

routes = pd.DataFrame(route_records)

print("\n4. ROUTES CREATED")
print("Routes:", len(routes))
print(routes.head())

In [ ]:
state_route_map = {}

for state in states:
    state_routes = routes[
        routes["state"] == state
    ]["route_id"].tolist()

    state_route_map[state] = state_routes

assigned_routes = []

for state in orders["customer_state"]:
    available_routes = state_route_map.get(state)

    if available_routes:
        assigned_routes.append(
            rng.choice(available_routes)
        )
    else:
        assigned_routes.append(
            routes.iloc[0]["route_id"]
        )

orders["route_id"] = assigned_routes

orders["driver_id"] = rng.choice(
    drivers["driver_id"],
    size=len(orders)
)

driver_vehicle_map = dict(
    zip(
        vehicles["driver_id"],
        vehicles["vehicle_id"]
    )
)

orders["vehicle_id"] = orders["driver_id"].map(
    driver_vehicle_map
)

print("\nASSIGNMENT COMPLETE")

print("\nUnique routes:", orders["route_id"].nunique())
print("Unique drivers:", orders["driver_id"].nunique())
print("Unique vehicles:", orders["vehicle_id"].nunique())

print("\nSample:")
print(
    orders[
        [
            "order_id",
            "customer_state",
            "route_id",
            "driver_id",
            "vehicle_id"
        ]
    ].head(10)
)

In [ ]:
issue_reasons = [
    "TRAFFIC",
    "CUSTOMER_UNAVAILABLE",
    "INCORRECT_ADDRESS",
    "VEHICLE_ISSUE",
    "WEATHER",
    "DELIVERY_TIME_EXCEEDED"
]

issue_probabilities = [
    0.32,
    0.27,
    0.16,
    0.08,
    0.07,
    0.10
]

# Generate a reason for records classified as problematic
problematic = orders["delivery_failure"] == 1

orders["issue_reason"] = "NONE"

orders.loc[problematic, "issue_reason"] = rng.choice(
    issue_reasons,
    size=problematic.sum(),
    p=issue_probabilities
)

print("\nISSUE REASONS CREATED")

print(
    orders["issue_reason"]
    .value_counts()
)

In [ ]:
attempt_records = []

for _, row in orders.iterrows():


    first_attempt_status = (
        "SUCCESS"
        if row["order_status"] == "delivered"
        else "FAILED"
    )

    attempt_records.append({
        "attempt_id": f"A{len(attempt_records) + 1:07d}",
        "order_id": row["order_id"],
        "driver_id": row["driver_id"],
        "route_id": row["route_id"],
        "attempt_number": 1,
        "attempt_status": first_attempt_status,
        "issue_reason": row["issue_reason"]
    })

    if (
        row["order_status"] == "delivered"
        and rng.random() < 0.08
    ):
        attempt_records.append({
            "attempt_id": f"A{len(attempt_records) + 1:07d}",
            "order_id": row["order_id"],
            "driver_id": row["driver_id"],
            "route_id": row["route_id"],
            "attempt_number": 2,
            "attempt_status": "SUCCESS",
            "issue_reason": "NONE"
        })

delivery_attempts = pd.DataFrame(attempt_records)

print("\nATTEMPTS CREATED")
print("Total attempts:", len(delivery_attempts))

print("\nAttempt status:")
print(
    delivery_attempts["attempt_status"]
    .value_counts()
)

print("\nAttempts per order:")
print(
    delivery_attempts
    .groupby("order_id")
    .size()
    .value_counts()
    .sort_index()
)

In [ ]:
SYNTHETIC = ROOT / "data" / "synthetic"
SYNTHETIC.mkdir(parents=True, exist_ok=True)

print("\nSynthetic folder:")
print(SYNTHETIC)

In [ ]:
drivers.to_csv(
    SYNTHETIC / "drivers.csv",
    index=False
)

vehicles.to_csv(
    SYNTHETIC / "vehicles.csv",
    index=False
)

routes.to_csv(
    SYNTHETIC / "routes.csv",
    index=False
)

delivery_attempts.to_csv(
    SYNTHETIC / "delivery_attempts.csv",
    index=False
)

orders.to_csv(
    SYNTHETIC / "routesense_orders_enriched.csv",
    index=False
)

print("ALL SYNTHETIC DATA SAVED")

for file in SYNTHETIC.iterdir():
    print(file.name)